In [ ]:
# 03 - Decision Tree (TP4): Decision Tree classification on ourdataset + classification metrics + MLflow
#
# Like course 4 (the diabetes notebook), but on our heart sounds (ourdataset).
# The project plan says: "Decision Tree (Arbre de Décision) sur caractéristiques 1D (MFCC)"
#   -> for each wav we compute its MFCC and keep 1D features (1 list of 26 numbers per wav)
#
# Before running: the MLflow container must be running (docker start mlflow-server)

In [ ]:
import pickle
from glob import glob

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import librosa
import requests                                                  # only to check that MLflow is running
import IPython.display as ipd

from sklearn.model_selection import train_test_split, KFold
from sklearn.tree import DecisionTreeClassifier, plot_tree        # the decision tree + a function to draw it
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import confusion_matrix, classification_report

import mlflow
import mlflow.sklearn

In [ ]:
# my folders (the same as in the other notebooks)
project_folder = "D:/MLOps_Project/"
ourdataset_folder = "D:/MLOps_Project/ourdataset/train/"


def load_data(file_name):
    with open(file_name, "rb") as f:    # "rb" = read binary
        return pickle.load(f)


def save_data(data, file_name):
    with open(file_name, "wb") as f:    # "wb" = write binary
        pickle.dump(data, f)


# MLflow (Service 6): our server = the Docker container "mlflow-server", on port 5000
mlflow.set_tracking_uri("http://localhost:5000")
try:
    requests.get("http://localhost:5000/health", timeout=3)    # is the server there?
    mlflow.set_experiment("audio_benchmark")    # the same experiment as the SVM (TP3), to compare the models
    print("MLflow server: OK")
except requests.exceptions.ConnectionError:
    print("MLflow server NOT running -> run 'docker start mlflow-server' in the terminal, then run this cell again")

In [ ]:
# ==================== The features: the MFCC of each wav ====================
# MFCC = a summary of the Mel spectrogram (course 2): log of the Mel spectrogram + one more math step (DCT)
#        -> 13 numbers per moment that describe the "shape" of the sound (the classic features for sounds)
SR = 2000          # every wav at 2000 Hz (like TP3 and the project plan)
SECONDS = 5        # every wav = 5 seconds (the project plan: "normalisation 2s ou 5s")
N_MFCC = 13        # 13 MFCC coefficients (the classic number)

# the same cleaning rules as TP3 (STEP 2), so we keep the same wavs as the SVM
MIN_DURATION = 2       # seconds
MIN_PEAK = 0.001       # almost no sound
MAX_CLIPPING = 1       # % of saturated samples


def get_mfcc_features(file):
    signal, rate = librosa.load(file, sr=SR)

    # 1. cleaning: a broken wav gives None (we will skip it)
    duration = len(signal) / rate
    peak = np.max(np.abs(signal))
    clipping_pct = 100 * np.mean(np.abs(signal) >= 0.99)
    if duration < MIN_DURATION or peak < MIN_PEAK or clipping_pct > MAX_CLIPPING:
        return None

    # 2. the same duration for every wav (like TP3): the middle 5 s, a shorter wav is repeated
    n_samples = SR * SECONDS
    if len(signal) < n_samples:
        repeat = int(np.ceil(n_samples / len(signal)))
        signal = np.tile(signal, repeat)[:n_samples]
    else:
        start = (len(signal) - n_samples) // 2
        signal = signal[start:start + n_samples]

    # 3. MFCC: a table of 13 rows (the coefficients) x 1 column per moment (every 0.064 s)
    mfcc = librosa.feature.mfcc(y=signal, sr=rate, n_mfcc=N_MFCC, n_fft=256, hop_length=128, n_mels=30)

    # 4. 1D features: the mean and the std of each coefficient over the 5 s -> 13 + 13 = 26 numbers
    return list(np.mean(mfcc, axis=1)) + list(np.std(mfcc, axis=1))


def extractMfcc_features():
    a = []    # the features: 1 list of 26 numbers per wav
    b = []    # the class of each wav: 1 = abnormal, 0 = normal
    removed = 0
    for nametype in ["abnormal", "normal"]:
        for file in glob(ourdataset_folder + nametype + "/*.wav"):
            features = get_mfcc_features(file)
            if features is None:          # broken wav -> we skip it
                removed = removed + 1
                continue
            a.append(features)
            if nametype == "normal":
                b.append(0)
            else:
                b.append(1)

    # saved like the prof's files: a = the features, b = the classes
    save_data(a, project_folder + "mfcc_features_tree_a.data")
    save_data(b, project_folder + "mfcc_features_tree_b.data")
    print("removed", removed, "broken wavs | kept", len(a), "wavs x", len(a[0]), "MFCC features")
    return a, b

In [ ]:
# reads all the wavs (a few minutes). Run it once: after that, the 2 .data files are enough
a, b = extractMfcc_features()

In [ ]:
a = load_data(project_folder + "mfcc_features_tree_a.data")
b = load_data(project_folder + "mfcc_features_tree_b.data")

X = np.array(a)    # 1 row per wav, 26 columns
y = np.array(b)    # 0 = normal, 1 = abnormal

# the names of the 26 features: mfcc1_mean ... mfcc13_mean, then mfcc1_std ... mfcc13_std
feature_names = []
for i in range(1, N_MFCC + 1):
    feature_names.append("mfcc" + str(i) + "_mean")
for i in range(1, N_MFCC + 1):
    feature_names.append("mfcc" + str(i) + "_std")

print("dimension of our data:", X.shape)
print("normal (0):", np.sum(y == 0), "| abnormal (1):", np.sum(y == 1))

sns.countplot(x=y)    # like the course: how many wavs in each class
plt.xticks([0, 1], ["normal (0)", "abnormal (1)"])
plt.show()

pd.DataFrame(X, columns=feature_names).head()

In [ ]:
# The formulas of the course, on our data (before the first question of the tree)
p_normal = np.mean(y == 0)
p_abnormal = np.mean(y == 1)

gini = 1 - p_normal ** 2 - p_abnormal ** 2                                    # Gini impurity
entropy = -(p_normal * np.log2(p_normal) + p_abnormal * np.log2(p_abnormal))  # entropy (ID3, Quinlan)

print("p(normal) = {:.3f} | p(abnormal) = {:.3f}".format(p_normal, p_abnormal))
print("Gini impurity = {:.3f}   (0 = only 1 class, 0.5 = 50/50)".format(gini))
print("Entropy       = {:.3f}   (0 = only 1 class, 1 = 50/50)".format(entropy))
# The tree looks for the question "feature <= threshold" (like "Weight < 167.5" in the course)
# that makes the 2 groups the most pure (the biggest gain), then again in each group, and so on

In [ ]:
# like the course: 67% to train, 33% to test, stratified (the same % of abnormal in both parts)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, stratify=y, random_state=66)
print("train:", X_train.shape[0], "wavs | test:", X_test.shape[0], "wavs")

In [ ]:
# 1. A tree without any limit (like the course)
# class_weight="balanced": we have 3.5x more normal wavs than abnormal ones, so an abnormal wav counts 3.5x more
# (the same as the SVM in TP3)
tree = DecisionTreeClassifier(class_weight="balanced", random_state=0)
tree.fit(X_train, y_train)
print("Accuracy on training set: {:.3f}".format(tree.score(X_train, y_train)))
print("Accuracy on test set: {:.3f}".format(tree.score(X_test, y_test)))
print("depth of the tree:", tree.get_depth(), "| number of leaves:", tree.get_n_leaves())
# 100% on the train set but much less on the test set = OVERFITTING: the tree learned the train wavs
# by heart (a lot of tiny leaves) -> we must limit its depth (pre-pruning, like the course)

In [ ]:
# 2. Which max_depth? And which criterion: "gini" (Gini impurity) or "entropy" (ID3)? Both are in the course.
#    We try max_depth = 1 to 15 with both, and look at the test set
rows = []
for criterion in ["gini", "entropy"]:
    for depth in range(1, 16):
        tree = DecisionTreeClassifier(criterion=criterion, max_depth=depth, class_weight="balanced", random_state=0)
        tree.fit(X_train, y_train)
        y_pred = tree.predict(X_test)
        rows.append({
            "criterion": criterion,
            "max_depth": depth,
            "train_accuracy": tree.score(X_train, y_train),
            "test_accuracy": accuracy_score(y_test, y_pred),
            "test_recall": recall_score(y_test, y_pred),    # recall and f1: of the abnormal class
            "test_f1": f1_score(y_test, y_pred),
        })
search = pd.DataFrame(rows)

# the best tree = the highest F1 of abnormal on the test set
# (F1 balances the recall = find the sick hearts, and the precision = not too many false alarms)
best = search.loc[search["test_f1"].idxmax()]
best_criterion = best["criterion"]
best_depth = int(best["max_depth"])
print("Best tree: criterion =", best_criterion, "| max_depth =", best_depth,
      "| test F1 = {:.3f} | test recall = {:.3f}".format(best["test_f1"], best["test_recall"]))

# ONE figure: left = the overfitting (train vs test accuracy), right = the F1 of abnormal (gini vs entropy)
plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
gini_rows = search[search["criterion"] == "gini"]
plt.plot(gini_rows["max_depth"], gini_rows["train_accuracy"], marker="o", label="train")
plt.plot(gini_rows["max_depth"], gini_rows["test_accuracy"], marker="o", label="test")
plt.title("Accuracy (gini): the deeper the tree, the more overfitting")
plt.xlabel("max_depth")
plt.ylabel("Accuracy")
plt.legend()

plt.subplot(1, 2, 2)
for criterion in ["gini", "entropy"]:
    criterion_rows = search[search["criterion"] == criterion]
    plt.plot(criterion_rows["max_depth"], criterion_rows["test_f1"], marker="o", label=criterion)
plt.axvline(best_depth, color="black", linestyle="--", linewidth=1)    # the best max_depth
plt.title("F1 of abnormal on the test set (dashed line = best)")
plt.xlabel("max_depth")
plt.ylabel("F1")
plt.legend()

plt.tight_layout()
plt.savefig(project_folder + "tp4_choose_depth.png")
plt.show()

In [ ]:
# 3. The pruned tree (like max_depth=3 in the course), with the best settings
tree = DecisionTreeClassifier(criterion=best_criterion, max_depth=best_depth, class_weight="balanced", random_state=0)
tree.fit(X_train, y_train)
print("Accuracy on training set: {:.3f}".format(tree.score(X_train, y_train)))
print("Accuracy on test set: {:.3f}".format(tree.score(X_test, y_test)))

y_true, y_pred = y_test, tree.predict(X_test)
print(classification_report(y_true, y_pred, target_names=["normal", "abnormal"]))

In [ ]:
# 4. Feature importance (like the course): how much each feature helped the tree to decide
#    (a number between 0 and 1, the sum = 1, 0 = never used)
def plot_feature_importances(model):
    plt.figure(figsize=(8, 8))
    n_features = len(feature_names)
    plt.barh(range(n_features), model.feature_importances_, align="center")
    plt.yticks(np.arange(n_features), feature_names)
    plt.xlabel("Feature importance")
    plt.ylabel("Feature")
    plt.ylim(-1, n_features)


plot_feature_importances(tree)
plt.title("Feature importance (max_depth = " + str(best_depth) + ")")
plt.tight_layout()
plt.savefig(project_folder + "tp4_feature_importance.png")
plt.show()

importances = pd.Series(tree.feature_importances_, index=feature_names).sort_values(ascending=False)
print("The 5 most important features:")
print(importances.head(5).round(3))
print("Features never used by the tree:", np.sum(tree.feature_importances_ == 0), "of", len(feature_names))

In [ ]:
# 5. The tree itself (only the first 3 levels, the whole tree is too big to read)
# In each box: the question (like "Weight < 167.5" in the course), the gini (or entropy) of the box,
# samples = how many train wavs arrive in the box, value = [normal, abnormal] in the box (weighted),
# class = what the tree answers for them. True = the question is true -> go left.
# (with class_weight="balanced" the 2 classes weigh the same, so the first box is 50/50: gini = 0.5 or entropy = 1)
plt.figure(figsize=(22, 9))
plot_tree(tree, max_depth=2, feature_names=feature_names, class_names=["normal", "abnormal"],
          filled=True, rounded=True, fontsize=9)
plt.title("The first levels of our decision tree")
plt.savefig(project_folder + "tp4_tree.png")
plt.show()

In [ ]:
# 6. The 3 folds (like TP3): each fold, 1 third of the wavs = test (33%), the 2 other thirds = train (66%)
#    The same KFold as TP3 on the same wavs -> the same 3 folds as the SVM: fair comparison in MLflow
kfold = KFold(n_splits=3, shuffle=True, random_state=1)

results = []               # the metrics of each fold
confusion_matrices = []    # kept to draw them after
fold = 1
for train_index, test_index in kfold.split(X):
    X_train, X_test = X[train_index], X[test_index]
    y_train, y_test = y[train_index], y[test_index]

    tree = DecisionTreeClassifier(criterion=best_criterion, max_depth=best_depth, class_weight="balanced", random_state=0)
    tree.fit(X_train, y_train)
    y_pred = tree.predict(X_test)

    print("--- Fold", fold, ": train =", len(train_index), "wavs | test =", len(test_index), "wavs")
    print(classification_report(y_test, y_pred, target_names=["normal", "abnormal"]))

    results.append({
        "fold": fold,
        "accuracy": accuracy_score(y_test, y_pred),     # % of good answers
        "precision": precision_score(y_test, y_pred),   # these 3: for the abnormal class (1)
        "recall": recall_score(y_test, y_pred),
        "f1": f1_score(y_test, y_pred),
    })
    confusion_matrices.append(confusion_matrix(y_test, y_pred))
    fold = fold + 1

table = pd.DataFrame(results)
ipd.display(table.round(3))
print("Mean of the 3 folds:")
print(table[["accuracy", "precision", "recall", "f1"]].mean().round(3))

In [ ]:
def PlotConfusionMatrix(cm, title):
    # cm[i][j] = number of test wavs of class i (true) that the tree put in class j (predicted)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Greens", cbar=False,
                xticklabels=["normal", "abnormal"], yticklabels=["normal", "abnormal"])
    plt.title(title)
    plt.xlabel("Predicted class")
    plt.ylabel("True class")


# ONE figure: the confusion matrix of each fold + the metrics of each fold
plt.figure(figsize=(22, 5))
for i in range(3):
    plt.subplot(1, 4, i + 1)
    PlotConfusionMatrix(confusion_matrices[i],
                        "Fold " + str(i + 1) + " - accuracy=" + str(round(results[i]["accuracy"], 2)))

plt.subplot(1, 4, 4)
x = np.arange(3)    # 0, 1, 2 = fold 1, 2, 3
plt.bar(x - 0.25, table["accuracy"], width=0.25, label="accuracy")
plt.bar(x, table["recall"], width=0.25, label="recall (abnormal)")
plt.bar(x + 0.25, table["f1"], width=0.25, label="f1 (abnormal)")
plt.xticks(x, ["Fold 1", "Fold 2", "Fold 3"])
plt.ylim(0, 1.15)    # a bit of space above the bars for the legend
plt.title("Decision tree: metrics of each fold")
plt.legend(loc="upper center", ncol=3, fontsize=8)

plt.tight_layout()
plt.savefig(project_folder + "tp4_tree_figure.png")
plt.show()

In [ ]:
# ==================== MLflow (Service 6): this model in its own run ====================
with mlflow.start_run(run_name="decision_tree_" + best_criterion + "_depth" + str(best_depth)):
    # 1. params = the settings we chose
    mlflow.log_param("model", "DecisionTree")
    mlflow.log_param("criterion", best_criterion)
    mlflow.log_param("max_depth", best_depth)
    mlflow.log_param("class_weight", "balanced")
    mlflow.log_param("n_folds", 3)
    mlflow.log_param("features", "MFCC mean + std (26)")
    mlflow.log_param("seconds", SECONDS)
    mlflow.log_param("sample_rate", SR)
    mlflow.log_param("n_features", X.shape[1])
    mlflow.log_param("n_wavs", len(y))

    # 2. metrics = the mean of the 3 folds (the same names as the SVM runs, to compare them)
    for metric in ["accuracy", "precision", "recall", "f1"]:
        mlflow.log_metric("mean_" + metric, table[metric].mean())
    for r in results:
        mlflow.log_metric("fold_accuracy", r["accuracy"], step=r["fold"])
        mlflow.log_metric("fold_recall", r["recall"], step=r["fold"])

    # 3. artifacts = the figures of this notebook
    mlflow.log_artifact(project_folder + "tp4_choose_depth.png")
    mlflow.log_artifact(project_folder + "tp4_feature_importance.png")
    mlflow.log_artifact(project_folder + "tp4_tree.png")
    mlflow.log_artifact(project_folder + "tp4_tree_figure.png")

    # 4. the model. The 3 folds were only to MEASURE the tree: the tree we keep is trained on ALL the wavs
    final_tree = DecisionTreeClassifier(criterion=best_criterion, max_depth=best_depth,
                                        class_weight="balanced", random_state=0)
    final_tree.fit(X, y)
    mlflow.sklearn.log_model(final_tree, "model", input_example=X[:5])

In [ ]:
# All the runs of the experiment (the SVM runs of TP3 + this tree), the highest recall first:
# the project keeps the model with the highest recall of abnormal (the sick hearts)
runs = mlflow.search_runs(experiment_names=["audio_benchmark"])
columns = ["tags.mlflow.runName", "params.model", "params.features", "metrics.mean_recall",
           "metrics.mean_f1", "metrics.mean_precision", "metrics.mean_accuracy"]
ipd.display(runs[columns].sort_values("metrics.mean_recall", ascending=False).round(3))